# XGBoost vs MLP -- NFL ATS Margin Prediction

**Experimental -- not production code.**

Trains two models on the same ~79-feature set used by the production `predict_betting.ipynb` model:

| Model | Architecture | Library |
|-------|-------------|--------|
| XGBoost | Gradient-boosted trees | xgboost |
| MLP | 3-layer feedforward NN | PyTorch |

Both predict the **home team margin of victory**. ATS accuracy is the primary business metric.


## Setup

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch', '--quiet'], check=False)

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import nflreadpy as nfl
import xgboost as xgb
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt

print(f'XGBoost {xgb.__version__}  |  PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')


## Configuration

In [ ]:
TRAIN_SEASONS = list(range(2014, 2024))  # 10 seasons
TEST_SEASONS  = [2024]                   # 1-season holdout
ROLL_N        = 5
ALLPRO_CSV    = Path('../betting/nfl_allpro_1997_2025.csv')
TEAM_MAP = {'STL':'LA','LAR':'LA','OAK':'LV','LVR':'LV','SD':'LAC','SDG':'LAC',
            'NWE':'NE','KAN':'KC','GNB':'GB','NOR':'NO','TAM':'TB','SFO':'SF'}
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


## Step 1 -- Load Raw Data

Schedules and play-by-play from nflreadpy. PBP is used for EPA, yards, sacks, turnovers, third-down rate, and QB passer rating.


In [ ]:
ALL_SEASONS = TRAIN_SEASONS + TEST_SEASONS

# Schedules
raw_sched = nfl.load_schedules(ALL_SEASONS)
sched = raw_sched.to_pandas() if hasattr(raw_sched, 'to_pandas') else pd.DataFrame(raw_sched)
sched = sched[sched['game_type'].isin(['REG','POST'])].copy()
sched['home_margin'] = sched['home_score'] - sched['away_score']
sched['gameday'] = pd.to_datetime(sched['gameday'])
sched = sched.sort_values(['season','week','gameday']).reset_index(drop=True)
print(f'Schedule: {len(sched):,} rows  |  {sched["season"].min()}-{sched["season"].max()}')

# Play-by-play
print('Loading PBP (takes ~1 min)...')
raw_pbp = nfl.load_pbp(ALL_SEASONS)
pbp = raw_pbp.to_pandas() if hasattr(raw_pbp, 'to_pandas') else pd.DataFrame(raw_pbp)
pbp = pbp[pbp['season_type'].isin(['REG','POST'])].copy()

# Aggregate per-game team stats from PBP
pbp_epa = pbp[pbp['epa'].notna()].copy()
off_stats = pbp_epa.groupby(['game_id','posteam']).agg(
    off_epa_game=('epa','mean'),
    off_yards_game=('yards_gained','mean'),
    off_plays=('play_id','count'),
).reset_index().rename(columns={'posteam':'team'})
def_stats = pbp_epa.groupby(['game_id','defteam']).agg(
    def_epa_game=('epa','mean'),
    def_yards_game=('yards_gained','mean'),
).reset_index().rename(columns={'defteam':'team'})

pbp['turnover'] = ((pbp['interception']==1)|(pbp['fumble_lost']==1)).astype(int)
pbp['third_att']  = (pbp['down']==3).astype(int)
pbp['third_conv'] = ((pbp['down']==3)&(pbp['first_down']==1)).astype(int)
adv_stats = pbp.groupby(['game_id','posteam']).agg(
    turnovers=('turnover','sum'),
    third_att=('third_att','sum'),
    third_conv=('third_conv','sum'),
).reset_index().rename(columns={'posteam':'team'})
adv_stats['third_down_rate'] = adv_stats['third_conv'] / adv_stats['third_att'].replace(0,1)

sack_stats = pbp[pbp['sack']==1].groupby(['game_id','defteam']).size().reset_index(name='sacks').rename(columns={'defteam':'team'})

team_pbp = off_stats.merge(def_stats, on=['game_id','team'], how='outer')
team_pbp = team_pbp.merge(adv_stats[['game_id','team','turnovers','third_down_rate']], on=['game_id','team'], how='left')
team_pbp = team_pbp.merge(sack_stats, on=['game_id','team'], how='left')
team_pbp = team_pbp.fillna(0)
print(f'Team-game PBP stats: {len(team_pbp):,} rows')


## Step 2 -- Feature Engineering (matching production)

All rolling features use `shift(1).rolling(n)` -- always from previous N games, no leakage.

**Feature groups:**
1. Rolling win%, scoring, EPA, yards
2. Cover rate (ATS history)
3. Sacks, turnovers, third-down conversion
4. Strength of schedule (rolling 3-game + season-long)
5. All-Pro quality (overall, offense, defense splits + prev year)
6. QB passer rating (prior season) + QB switch flag
7. Coach win % (cumulative prior to each game)
8. Game context (spread, weather, playoff flags)


In [ ]:
# Build long format: one row per team per game
home_rows = sched[['game_id','season','week','home_team','away_team','home_score','away_score','home_margin']].copy()
home_rows['team'] = home_rows['home_team']; home_rows['opponent'] = home_rows['away_team']
home_rows['is_home'] = 1
home_rows['pts_for'] = home_rows['home_score']; home_rows['pts_against'] = home_rows['away_score']
home_rows['won'] = (home_rows['home_margin'] > 0).astype(int)

away_rows = sched[['game_id','season','week','home_team','away_team','home_score','away_score','home_margin']].copy()
away_rows['team'] = away_rows['away_team']; away_rows['opponent'] = away_rows['home_team']
away_rows['is_home'] = 0
away_rows['pts_for'] = away_rows['away_score']; away_rows['pts_against'] = away_rows['home_score']
away_rows['won'] = (away_rows['home_margin'] < 0).astype(int)

tg = pd.concat([home_rows, away_rows], ignore_index=True)
tg = tg.merge(team_pbp, on=['game_id','team'], how='left')

# Cover rate (home covered = margin > spread; away covered = margin < spread)
spread_lkp = sched[['game_id','spread_line']].copy()
spread_lkp['spread_line'] = pd.to_numeric(spread_lkp['spread_line'], errors='coerce')
tg = tg.merge(spread_lkp, on='game_id', how='left')
tg['covered'] = np.where(tg['is_home']==1,
    (tg['home_margin'] > tg['spread_line']).astype(int),
    (tg['home_margin'] < tg['spread_line']).astype(int))

tg = tg.sort_values(['team','season','week']).reset_index(drop=True)

def roll(s, n=ROLL_N):
    return s.shift(1).rolling(n, min_periods=1).mean()

grp = tg.groupby('team')
tg['win_pct_roll']      = grp['won'].transform(roll)
tg['pts_for_roll']      = grp['pts_for'].transform(roll)
tg['pts_against_roll']  = grp['pts_against'].transform(roll)
tg['off_epa_roll']      = grp['off_epa_game'].transform(roll)
tg['def_epa_roll']      = grp['def_epa_game'].transform(roll)
tg['off_yards_roll']    = grp['off_yards_game'].transform(roll)
tg['def_yards_roll']    = grp['def_yards_game'].transform(roll)
tg['cover_rate_roll']   = grp['covered'].transform(roll)
tg['sacks_roll']        = grp['sacks'].transform(roll)
tg['turnovers_roll']    = grp['turnovers'].transform(roll)
tg['third_down_roll']   = grp['third_down_rate'].transform(roll)
print('Basic rolling features built.')


In [ ]:
# Strength of schedule: opponent win% rolling-3 and season-long
opp_lkp = tg[['game_id','team','win_pct_roll']].rename(columns={'team':'opponent','win_pct_roll':'opp_cur_win_pct'})
tg = tg.merge(opp_lkp, on=['game_id','opponent'], how='left')

grp = tg.groupby('team')
tg['sos_roll3']   = grp['opp_cur_win_pct'].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
tg['sos_season']  = grp['opp_cur_win_pct'].transform(lambda s: s.shift(1).expanding().mean())
print('SOS features built.')


In [ ]:
if ALLPRO_CSV.exists():
    ap = pd.read_csv(ALLPRO_CSV)
    ap.rename(columns={'Year':'ap_season','Team':'ap_team','Side':'ap_side'}, inplace=True)
    ap['ap_team'] = ap['ap_team'].replace(TEAM_MAP)
    ap = ap[ap['ap_team'] != '2TM']

    def build_weighted_ap(df_ap):
        frames = []
        for yr in range(2000, 2027):
            rows = []
            for lag, w in {0:4, 1:2, 2:1}.items():
                tmp = df_ap[df_ap['ap_season'] == yr-lag-1].copy()
                tmp['w'] = w; tmp['yr_target'] = yr
                rows.append(tmp)
            if not rows: continue
            comb = pd.concat(rows)
            deduped = comb.sort_values('w', ascending=False).drop_duplicates(['Player','yr_target'])
            wc = deduped.groupby(['yr_target','ap_team'])['w'].sum().reset_index()
            wc.columns = ['season','team','val']
            frames.append(wc)
        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=['season','team','val'])

    overall_ap = build_weighted_ap(ap)
    off_ap     = build_weighted_ap(ap[ap['ap_side']=='offense'])
    def_ap     = build_weighted_ap(ap[ap['ap_side']=='defense'])
    prev_ap    = ap.groupby(['ap_season','ap_team'])['Player'].nunique().reset_index()
    prev_ap.columns = ['season','team','val']
    prev_ap['season'] = prev_ap['season'] + 1  # shift to target season

    def merge_ap(df, lookup, col):
        lkp = lookup.rename(columns={'val': col})
        df = df.merge(lkp, on=['season','team'], how='left')
        df[col] = df[col].fillna(0)
        return df

    tg = merge_ap(tg, overall_ap, 'allpro_overall')
    tg = merge_ap(tg, off_ap,     'allpro_off')
    tg = merge_ap(tg, def_ap,     'allpro_def')
    tg = merge_ap(tg, prev_ap,    'allpro_prev')
    print('All-Pro features added.')
else:
    for col in ['allpro_overall','allpro_off','allpro_def','allpro_prev']:
        tg[col] = 0.0
    print('All-Pro CSV not found.')


In [ ]:
# QB passer rating (prior season) from PBP
qb_pr = {}
for s in ALL_SEASONS:
    pp = pbp[(pbp['play_type']=='pass') & (pbp['season']==s) & pbp['passer_player_name'].notna()].copy()
    if pp.empty: continue
    qbs = pp.groupby(['posteam','passer_player_name']).agg(
        att=('pass_attempt','sum'), comp=('complete_pass','sum'),
        yds=('passing_yards','sum'), tds=('pass_touchdown','sum'), ints=('interception','sum')
    ).reset_index()
    qbs = qbs[qbs['att'] >= 100].copy()
    def pr(r):
        a = max(0, min(((r['comp']/r['att'])-0.3)*5,    2.375))
        b = max(0, min(((r['yds']/r['att'])-3)*0.25,    2.375))
        c = max(0, min(r['tds']/r['att']*20,            2.375))
        d = max(0, min(2.375-(r['ints']/r['att']*25),   2.375))
        return ((a+b+c+d)/6)*100
    qbs['pr'] = qbs.apply(pr, axis=1)
    best = qbs.sort_values('att', ascending=False).groupby('posteam').first()[['pr']].reset_index()
    for _, row in best.iterrows():
        qb_pr[(s+1, row['posteam'])] = row['pr']  # available in NEXT season

median_pr = float(np.median(list(qb_pr.values()))) if qb_pr else 85.0
tg['qbr_prev'] = tg.apply(lambda r: qb_pr.get((r['season'], r['team']), median_pr), axis=1)

# QB switch flag
if 'home_qb_name' in sched.columns and sched['home_qb_name'].notna().any():
    qb_rows = []
    for _, g in sched.iterrows():
        qb_rows += [{'game_id':g['game_id'],'team':g['home_team'],'qb':g.get('home_qb_name')},
                    {'game_id':g['game_id'],'team':g['away_team'],'qb':g.get('away_qb_name')}]
    qb_df = pd.DataFrame(qb_rows)
    qb_df = qb_df.merge(tg[['game_id','team','season','week']].drop_duplicates(), on=['game_id','team'], how='left')
    qb_df = qb_df.sort_values(['team','season','week'])
    qb_df['last_qb'] = qb_df.groupby('team')['qb'].shift(1)
    qb_df['qb_switch'] = ((qb_df['qb'] != qb_df['last_qb']) & qb_df['qb'].notna() & qb_df['last_qb'].notna()).astype(int)
    tg = tg.merge(qb_df[['game_id','team','qb_switch']], on=['game_id','team'], how='left')
    tg['qb_switch'] = tg['qb_switch'].fillna(0)
    print('QB switch flag added.')
else:
    tg['qb_switch'] = 0
    print('QB name not in schedule -- qb_switch set to 0.')

# Coach win pct (cumulative, prior to each game)
if 'home_coach' in sched.columns and sched['home_coach'].notna().any():
    coach_rows = []
    for _, g in sched.iterrows():
        hw = 1 if (g.get('home_score',0) or 0) > (g.get('away_score',0) or 0) else 0
        coach_rows += [
            {'game_id':g['game_id'],'season':g['season'],'week':g['week'],'team':g['home_team'],'coach':g.get('home_coach'),'won':hw},
            {'game_id':g['game_id'],'season':g['season'],'week':g['week'],'team':g['away_team'],'coach':g.get('away_coach'),'won':1-hw},
        ]
    cdf = pd.DataFrame(coach_rows).sort_values(['coach','season','week'])
    cdf['cum_wins']  = cdf.groupby('coach')['won'].cumsum().shift(fill_value=0)
    cdf['cum_games'] = cdf.groupby('coach').cumcount()
    cdf['coach_win_pct'] = (cdf['cum_wins'] / cdf['cum_games'].replace(0, np.nan)).fillna(0)
    tg = tg.merge(cdf[['game_id','team','coach_win_pct']], on=['game_id','team'], how='left')
    tg['coach_win_pct'] = tg['coach_win_pct'].fillna(0)
    print('Coach win pct added.')
else:
    tg['coach_win_pct'] = 0.0
    print('Coach not in schedule -- coach_win_pct set to 0.')


In [ ]:
# Pivot: one row per game with home_ and away_ prefixes
TEAM_FEATS = [
    'win_pct_roll','pts_for_roll','pts_against_roll',
    'off_epa_roll','def_epa_roll','off_yards_roll','def_yards_roll',
    'cover_rate_roll','sacks_roll','turnovers_roll','third_down_roll',
    'sos_roll3','sos_season',
    'allpro_overall','allpro_off','allpro_def','allpro_prev',
    'qbr_prev','qb_switch','coach_win_pct',
]

home_tg = tg[tg['is_home']==1][['game_id','season','week','home_team','away_team','home_margin'] + TEAM_FEATS].copy()
home_tg = home_tg.rename(columns={f:'h_'+f for f in TEAM_FEATS})

away_tg = tg[tg['is_home']==0][['game_id'] + TEAM_FEATS].copy()
away_tg = away_tg.rename(columns={f:'a_'+f for f in TEAM_FEATS})

games = home_tg.merge(away_tg, on='game_id')

# Weather + game context
extra = sched[['game_id','season','spread_line','total_line','roof','surface','temp','wind','game_type','week']].copy()
extra['is_turf']    = extra['surface'].str.lower().str.contains('turf|astro|field', na=False).astype(float)
extra['is_dome']    = extra['roof'].str.lower().isin(['dome','closed','retractable']).astype(float)
extra['is_playoff'] = (extra['game_type'] != 'REG').astype(int)
final_wks = sched[sched['game_type']=='REG'].groupby('season')['week'].max().rename('final_week')
extra = extra.join(final_wks, on='season')
extra['is_final_week'] = ((extra['game_type']=='REG') & (extra['week']==extra['final_week'])).astype(int)
extra['temp']        = pd.to_numeric(extra['temp'],        errors='coerce').fillna(65)
extra['wind']        = pd.to_numeric(extra['wind'],        errors='coerce').fillna(5)
extra['spread_line'] = pd.to_numeric(extra['spread_line'], errors='coerce')
extra['total_line']  = pd.to_numeric(extra['total_line'],  errors='coerce')

games = games.merge(
    extra[['game_id','spread_line','total_line','is_turf','is_dome','temp','wind','is_playoff','is_final_week']],
    on='game_id'
)

# Differential features (home minus away)
DIFF_FEATS = ['win_pct_roll','pts_for_roll','pts_against_roll','off_epa_roll','def_epa_roll',
              'off_yards_roll','def_yards_roll','cover_rate_roll','sacks_roll','turnovers_roll',
              'third_down_roll','allpro_overall','allpro_off','allpro_def','qbr_prev','coach_win_pct']
for f in DIFF_FEATS:
    games[f'd_{f}'] = games[f'h_{f}'] - games[f'a_{f}']

games = games.dropna(subset=['home_margin','spread_line']).reset_index(drop=True)
print(f'Game dataset: {len(games):,} rows  |  seasons {games["season"].min()}-{games["season"].max()}')


## Step 3 -- Train / Test Split

Split on season boundaries -- never shuffle across seasons.


In [ ]:
FEATURE_COLS = [
    # Home rolling
    'h_win_pct_roll','h_pts_for_roll','h_pts_against_roll',
    'h_off_epa_roll','h_def_epa_roll','h_off_yards_roll','h_def_yards_roll',
    'h_cover_rate_roll','h_sacks_roll','h_turnovers_roll','h_third_down_roll',
    'h_sos_roll3','h_sos_season',
    'h_allpro_overall','h_allpro_off','h_allpro_def','h_allpro_prev',
    'h_qbr_prev','h_qb_switch','h_coach_win_pct',
    # Away rolling
    'a_win_pct_roll','a_pts_for_roll','a_pts_against_roll',
    'a_off_epa_roll','a_def_epa_roll','a_off_yards_roll','a_def_yards_roll',
    'a_cover_rate_roll','a_sacks_roll','a_turnovers_roll','a_third_down_roll',
    'a_sos_roll3','a_sos_season',
    'a_allpro_overall','a_allpro_off','a_allpro_def','a_allpro_prev',
    'a_qbr_prev','a_qb_switch','a_coach_win_pct',
    # Differentials (home minus away)
    'd_win_pct_roll','d_pts_for_roll','d_pts_against_roll',
    'd_off_epa_roll','d_def_epa_roll','d_off_yards_roll','d_def_yards_roll',
    'd_cover_rate_roll','d_sacks_roll','d_turnovers_roll','d_third_down_roll',
    'd_allpro_overall','d_allpro_off','d_allpro_def','d_qbr_prev','d_coach_win_pct',
    # Game context
    'spread_line','total_line','is_turf','is_dome','temp','wind',
    'is_playoff','is_final_week','h_qb_switch','a_qb_switch',
]
TARGET = 'home_margin'

train_mask  = games['season'].isin(TRAIN_SEASONS)
test_mask   = games['season'].isin(TEST_SEASONS)
X_train     = games.loc[train_mask, FEATURE_COLS].fillna(0).values.astype('float32')
y_train     = games.loc[train_mask, TARGET].values.astype('float32')
X_test      = games.loc[test_mask,  FEATURE_COLS].fillna(0).values.astype('float32')
y_test      = games.loc[test_mask,  TARGET].values.astype('float32')
spread_test = games.loc[test_mask, 'spread_line'].values

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}  |  Features: {len(FEATURE_COLS)}')


## Step 4 -- XGBoost Baseline


In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=500, learning_rate=0.01, max_depth=3,
    subsample=0.6, colsample_bytree=0.6, min_child_weight=3,
    reg_alpha=1.0, reg_lambda=3.0, random_state=SEED, verbosity=0,
)
xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

xgb_preds = xgb_model.predict(X_test)
xgb_mae   = mean_absolute_error(y_test, xgb_preds)
xgb_rmse  = mean_squared_error(y_test, xgb_preds)**0.5
xgb_ats   = float(np.mean(np.sign(xgb_preds - spread_test) == np.sign(y_test - spread_test)))
print(f'XGBoost  |  MAE: {xgb_mae:.2f}  RMSE: {xgb_rmse:.2f}  ATS: {xgb_ats*100:.1f}%')

feat_imp = pd.Series(xgb_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8,5))
feat_imp.head(15).plot(kind='bar', ax=ax, color='#4a90d9')
ax.set_title('XGBoost -- Top 15 Feature Importances')
ax.set_ylabel('Importance'); plt.tight_layout(); plt.show()


## Step 5 -- MLP (PyTorch)

```
Input(N) -> Linear(256) -> BN -> ReLU -> Dropout(0.3)
         -> Linear(128) -> BN -> ReLU -> Dropout(0.2)
         -> Linear(64)  -> BN -> ReLU
         -> Linear(1)
```


In [ ]:
class BettingMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128),       nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64),        nn.BatchNorm1d(64),  nn.ReLU(),
            nn.Linear(64, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

mlp = BettingMLP(len(FEATURE_COLS)).to(DEVICE)
print(f'Trainable parameters: {sum(p.numel() for p in mlp.parameters() if p.requires_grad):,}')


In [ ]:
BATCH_SIZE = 64
EPOCHS     = 150
LR         = 3e-4

train_ds = TensorDataset(torch.tensor(X_train_sc, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32))
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

criterion = nn.HuberLoss(delta=7.0)
optimiser = torch.optim.AdamW(mlp.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=EPOCHS)

train_losses = []
for epoch in range(1, EPOCHS + 1):
    mlp.train()
    ep_loss = 0.0
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimiser.zero_grad()
        loss = criterion(mlp(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(mlp.parameters(), 1.0)
        optimiser.step()
        ep_loss += loss.item() * len(xb)
    scheduler.step()
    train_losses.append(ep_loss / len(train_ds))
    if epoch % 30 == 0 or epoch == EPOCHS:
        mlp.eval()
        with torch.no_grad():
            pv = mlp(torch.tensor(X_test_sc, dtype=torch.float32).to(DEVICE)).cpu().numpy()
        print(f'Epoch {epoch:>3}/{EPOCHS}  loss: {train_losses[-1]:.3f}  val_MAE: {mean_absolute_error(y_test,pv):.2f}')


In [ ]:
mlp.eval()
with torch.no_grad():
    mlp_preds = mlp(torch.tensor(X_test_sc, dtype=torch.float32).to(DEVICE)).cpu().numpy()

mlp_mae  = mean_absolute_error(y_test, mlp_preds)
mlp_rmse = mean_squared_error(y_test, mlp_preds)**0.5
mlp_ats  = float(np.mean(np.sign(mlp_preds - spread_test) == np.sign(y_test - spread_test)))
print(f'MLP      |  MAE: {mlp_mae:.2f}  RMSE: {mlp_rmse:.2f}  ATS: {mlp_ats*100:.1f}%')

plt.figure(figsize=(8,3))
plt.plot(train_losses, color='#e67e22')
plt.title('MLP Training Loss'); plt.xlabel('Epoch'); plt.ylabel('Huber Loss')
plt.tight_layout(); plt.show()


## Step 6 -- Head-to-Head Comparison


In [ ]:
vegas_mae  = mean_absolute_error(y_test, spread_test)
vegas_rmse = mean_squared_error(y_test,  spread_test)**0.5

results = pd.DataFrame({
    'Model':  ['Vegas Spread (baseline)', 'XGBoost', 'MLP (PyTorch)'],
    'MAE':    [round(vegas_mae,2),  round(xgb_mae,2),  round(mlp_mae,2)],
    'RMSE':   [round(vegas_rmse,2), round(xgb_rmse,2), round(mlp_rmse,2)],
    'ATS %':  ['~50.0%', f'{xgb_ats*100:.1f}%', f'{mlp_ats*100:.1f}%'],
})
print(results.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,5))
for ax, preds, label, color in zip(axes,[xgb_preds,mlp_preds],['XGBoost','MLP'],['#4a90d9','#e67e22']):
    ax.scatter(y_test, preds, alpha=0.3, s=10, color=color)
    lo = min(y_test.min(), preds.min())-3
    hi = max(y_test.max(), preds.max())+3
    ax.plot([lo,hi],[lo,hi],'k--',lw=1,alpha=0.5)
    ax.set_xlim(lo,hi); ax.set_ylim(lo,hi)
    ax.set_xlabel('Actual Margin'); ax.set_ylabel('Predicted Margin')
    ax.set_title(f'{label}  MAE={mean_absolute_error(y_test,preds):.2f}')
plt.suptitle('Predicted vs Actual Home Margin -- 2024', fontsize=13)
plt.tight_layout(); plt.show()

xgb_err = xgb_preds - y_test
mlp_err = mlp_preds - y_test
fig, ax = plt.subplots(figsize=(9,4))
ax.hist(xgb_err, bins=40, alpha=0.6, label=f'XGBoost  bias={xgb_err.mean():+.2f}', color='#4a90d9')
ax.hist(mlp_err, bins=40, alpha=0.6, label=f'MLP      bias={mlp_err.mean():+.2f}', color='#e67e22')
ax.axvline(0, color='white', lw=1.5, ls='--')
ax.set_xlabel('Error (predicted - actual)'); ax.set_ylabel('Count')
ax.set_title('Prediction Error Distribution -- 2024')
ax.legend(); plt.tight_layout(); plt.show()


## Takeaways

### Why XGBoost usually wins on tabular NFL data

1. **Small dataset** -- ~3,000 training games. Neural networks need more data to generalise; trees do well here.
2. **Hand-engineered tabular features** -- no hidden spatial or temporal structure left for the MLP to discover that XGBoost cannot capture with splits.
3. **High-noise target** -- NFL margins have an inherent predictability floor (~12-14 pts MAE). Both models converge near the same floor.
4. **Interactions** -- XGBoost finds feature interactions efficiently via splits; the MLP needs wide layers + strong regularisation to avoid overfitting.

### When sequence models (LSTM / Transformer) would make sense

- Feeding **raw play-by-play sequences** so the model learns its own game representation
- Much larger cross-sport or cross-league datasets
- **In-game live prediction** where the running sequence of plays within a game matters

### Next experiments to try

- **Ensemble**: average XGBoost + MLP predictions and check if ATS accuracy improves over either alone
- Wider/deeper MLP or residual (skip) connections
- XGBoost with a larger `n_estimators` budget and early stopping
